In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [6]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase, HW1
from openai import OpenAI


openai_client = OpenAI()

assistant = HW1(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Response Usage: ResponseUsage(input_tokens=7135, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=115, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7250)
It keeps calling the model inside a `while True` loop.

Each iteration:
1. Sends the full `messages` history to the model.
2. Checks the response for any `function_call` items.
3. Runs those tool calls and appends the results to `messages`.
4. Repeats.

It stops when the model returns a response with **no function calls**:

```python
if has_function_calls == False:
    break
```

So the exit condition is simply: no tool calls this turn means the model is done.


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
chunks[0:3]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [9]:
index_chunk = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunk.fit(chunks)

In [10]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase, HW1
from openai import OpenAI


openai_client = OpenAI()

assistant_w_chunk = HW1(
    index=index_chunk,
    llm_client=openai_client,
)

answer = assistant_w_chunk.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Response Usage: ResponseUsage(input_tokens=2318, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=118, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2436)
It keeps calling the model inside a `while True` loop, and after each model response it checks whether there were any `function_call` items.

- If there is at least one function call, it runs the tool, appends the result, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, the loop ends when the model returns a final answer without asking for any more tools.


In [11]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [12]:
def search(query: str) -> dict[str, str]:
    """
    Search the LLM Zoomcamp Lesson database for the top 5 entries matching the given query.
    """
    return index_chunk.search(
        query,
        num_results=5
    )


agent_tools = Tools()
agent_tools.add_tool(search)

In [13]:
instructions = """ 
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

In [14]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [15]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [16]:
result.cost

CostInfo(input_cost=Decimal('0.00596775'), output_cost=Decimal('0.0021915'), total_cost=Decimal('0.00815925'))

Cost using non-chunk
CostInfo(input_cost=Decimal('0.0238185'), output_cost=Decimal('0.002529'), total_cost=Decimal('0.0263475'))

Cost using chunk
CostInfo(input_cost=Decimal('0.006171'), output_cost=Decimal('0.002106'), total_cost=Decimal('0.008277'))
CostInfo(input_cost=Decimal('0.00605025'), output_cost=Decimal('0.0021555'), total_cost=Decimal('0.00820575'))
CostInfo(input_cost=Decimal('0.00596775'), output_cost=Decimal('0.0021915'), total_cost=Decimal('0.00815925'))
